In [3]:
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [4]:
def file_loader(path):
  loader = DirectoryLoader(
    path, glob="*.pdf", loader_cls=PyPDFLoader
  )
  documents = loader.load()
  return documents

In [5]:
extracted_docs = file_loader(r"Data/")

In [6]:
def chunking_data(data):
  split_data = RecursiveCharacterTextSplitter(chunk_size= 500, chunk_overlap = 50)
  chunk_data = split_data.split_documents(data)
  return chunk_data

In [7]:
chunk_data = chunking_data(extracted_docs)
len(chunk_data)

2124

In [9]:
def get_embedding():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [10]:
embedding = get_embedding()

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_23140\2649802877.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5779.98it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
docs = FAISS.from_documents(documents=chunk_data, embedding=embedding )
docs

In [12]:
retriver = docs.as_retriever(search_type="similarity", search_kwargs={"k": 3})
output = retriver.invoke("What is Supervised Machine Learning")
output

[Document(id='a8ee7d41-4b83-47fd-8569-f3e718737d46', metadata={'producer': 'xdvipdfmx (20211117)', 'creator': 'LaTeX via pandoc', 'creationdate': '2025-01-23T13:03:17+00:00', 'source': 'Data\\Hands-On-Machine-Learning.pdf', 'total_pages': 290, 'page': 11, 'page_label': '12'}, page_content='Figure 1-5. A labeled training set for supervised learning (e.g., spam classifica-\ntion) 8 | Chapter 1: The Machine Learning Landscape\nA typical supervised learning task is classification. The spam filter is a good\nexample of this: it is trained with many example emails along with their class\n(spam or ham), and it must learn how to classify new emails. Another typical\ntask is to predict a target numeric value, such as the price of a car, given a set of'),
 Document(id='2785c61e-bfbe-40a2-ac69-e0693d83e88a', metadata={'producer': 'xdvipdfmx (20211117)', 'creator': 'LaTeX via pandoc', 'creationdate': '2025-01-23T13:03:17+00:00', 'source': 'Data\\Hands-On-Machine-Learning.pdf', 'total_pages': 290, 

In [13]:
import os
OPENAI_API_KEY= os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
llm = ChatOpenAI(model = "gpt-4o-mini", temperature=0.6)

In [16]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

In [17]:
# LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [19]:
system_prompt = (
    "You are an expert Data Scientist assistant for question-answering tasks. "
    "Use ONLY the following retrieved context to answer the question. "
    "If the answer is not in the context, say you don't know. "
    "Do NOT hallucinate. "
    "Keep the answer concise, maximum 3 sentences.\n\n"
    "Context:\n{context}"
)

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("user", "{input}")
])

In [21]:
question = "What is Machine Learning?"

rag_chain = (
    {
        "context": retriver,              # lấy context từ FAISS
        "input": RunnablePassthrough()    # giữ nguyên câu hỏi
    }
    | chat_prompt
    | llm
)

response = rag_chain.invoke(question)

print(response.content)

Machine Learning is the science (and art) of programming computers so they can learn from data. It encompasses a field of study that enables computers to improve their performance on tasks through experience.


In [22]:
question = "What is Tensorflow?"

response = rag_chain.invoke(question)

print(response.content)

TensorFlow is an open-source machine learning framework that offers capabilities for various applications such as natural language processing, recommender systems, and time series forecasting. Its core is similar to NumPy but includes GPU support and distributed computing features. TensorFlow's API revolves around tensors, which are multidimensional arrays.
